# 📊 Classical Machine Learning: Zero to Hero — A Guided Lab

Before deep learning, these algorithms **were** machine learning — and they still power most
production systems (fraud detection, credit scoring, recommendation ranking). This lab builds
each one's core math from scratch, then uses scikit-learn for the production-grade version.

**Beginner-first.** Every chapter explains the *concept* and *math* in plain language before any
code. Prerequisite: the NumPy Zero-to-Master lab.

**How this lab works** — 📖 Theory (detailed) → 🧠 Mental model → 🖼️ ASCII diagram →
🔬 Worked example → ⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. The ML problem: fitting a function to data
2. Linear regression (from scratch + scikit-learn)
3. Logistic regression (classification)
4. Decision trees
5. Ensembles: random forests & boosting (concept)
6. k-Nearest Neighbors
7. k-Means clustering
8. Support Vector Machines (concept)
9. Dimensionality reduction: PCA
10. 🏆 Capstone: a full classical-ML pipeline on real data


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
np.random.seed(0)
print("Ready.")

---
## Chapter 1 — The ML Problem: Fitting a Function to Data

📖 **Theory.** Supervised ML is: given examples of inputs `X` and known outputs `y`, find a
function `f` such that `f(X) ≈ y`, and that **generalizes** to new, unseen inputs. Every
algorithm in this lab is a different strategy for choosing `f` and fitting its parameters.

Two flavors:
- **Regression** — `y` is a continuous number (price, temperature).
- **Classification** — `y` is a category (spam/not-spam, species).

🖼️ **Diagram — the supervised learning setup**
```
 training data:  (X1, y1), (X2, y2), ..., (Xn, yn)
                          │
                          ▼
                  fit f(X) to minimize error
                          │
                          ▼
       new X ──► f(X) ──► predicted y   (must generalize, not memorize!)
```

🧠 **Mental model.** Fitting is "finding the parameters that make the model's guesses closest to
reality on data we've already seen" — while hoping (and testing!) that it also works on data it
hasn't seen. That gap between "fits training data" and "generalizes" is the central tension in ML.


In [ ]:
# A toy regression problem: predict y from x with some noise
np.random.seed(0)
X_toy = np.linspace(0, 10, 30)
y_toy = 3*X_toy + 5 + np.random.normal(0, 2, 30)   # true relationship: y = 3x + 5, plus noise
print("first 5 (x, y) pairs:")
for x, y in zip(X_toy[:5], y_toy[:5]):
    print(f"  x={x:.2f}  y={y:.2f}")
print("\nGoal: recover the underlying y = 3x + 5 relationship from noisy data.")

### ✏️ Your Turn 1.1
In a comment, explain the difference between a model that **memorizes** training data and one
that **generalizes** — and why only the latter is useful.

In [ ]:
# your explanation


✅ **Solution**
```python
# Memorizing = perfectly reproducing the training examples seen, often by being so
# flexible it fits noise too (overfitting) -- useless on new data.
# Generalizing = capturing the underlying pattern, so it also works on data it never saw.
# Only generalization matters, since real-world use always involves new inputs.
```

---
## Chapter 2 — Linear Regression

📖 **Theory.** Fit a straight line `ŷ = wx + b` that minimizes the **mean squared error (MSE)**
between predictions and actual values. Two ways to solve it:
- **Closed-form (Normal Equation):** `w = (XᵀX)⁻¹Xᵀy` — exact, one-shot solution using linear
  algebra (from the NumPy lab!).
- **Gradient descent:** iteratively nudge `w, b` downhill on the error surface (from the PyTorch
  lab) — needed when closed-form is too expensive (huge datasets) or the model isn't linear.

🖼️ **Diagram — fitting a line**
```
  y │      •
    │    •   •
    │  •   •    ╱── the fitted line minimizes
    │•    •   ╱     total squared vertical distance
    │   •   ╱       from each point to the line
    └──────────► x
```


In [ ]:
# Closed-form solution via the Normal Equation (pure NumPy -- no library needed)
X_design = np.column_stack([X_toy, np.ones(len(X_toy))])   # add a column of 1s for the bias
w_closed = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y_toy
print(f"closed-form: slope={w_closed[0]:.3f} (true 3), intercept={w_closed[1]:.3f} (true 5)")

# scikit-learn version (production-grade, handles more cases: regularization, multiple features)
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_toy.reshape(-1,1), y_toy)
print(f"sklearn:     slope={lr.coef_[0]:.3f}, intercept={lr.intercept_:.3f}")

preds = lr.predict(X_toy.reshape(-1,1))
mse = mean_squared_error(y_toy, preds)
print(f"MSE: {mse:.3f}")

⚡ **Pro tip.** The closed-form Normal Equation only works cleanly for **linear** models with
a reasonably-sized feature count (matrix inversion is expensive at huge scale). Everything more
complex — logistic regression, neural nets — uses gradient descent instead, exactly like you
practiced in the PyTorch lab.

### ✏️ Your Turn 2.1
Fit a `LinearRegression` on the retail-style relationship: `units` predicting `revenue` (make up
a small synthetic array). Report the learned slope.

In [ ]:
units = np.array([1,2,3,4,5,6,7,8])
revenue = units * 25 + np.random.normal(0, 3, 8)   # price ~25 per unit + noise
slope = None
print(slope)

✅ **Solution**
```python
model = LinearRegression()
model.fit(units.reshape(-1,1), revenue)
slope = model.coef_[0]   # should be close to 25
```

---
## Chapter 3 — Logistic Regression (Classification)

📖 **Theory.** For classification, we don't want an unbounded number — we want a
**probability** (0 to 1). Logistic regression applies the **sigmoid function**
`σ(z) = 1/(1+e^-z)` to squash a linear combination `z = wx+b` into (0,1), then thresholds at 0.5
to decide the class. It's trained by minimizing **log loss** (cross-entropy — the same loss from
the PyTorch/TF classification chapters).

🖼️ **Diagram — the sigmoid squashing function**
```
 σ(z)
   1 ┤              ─────────
     │           ╱
 0.5 ┤        ╱          ← decision boundary: σ(z)=0.5 means z=0
     │     ╱
   0 ┤─────
     └──────────────────► z (any real number in, probability out)
```


In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

# a toy binary classification: pass/fail based on hours studied
hours = np.array([1,2,3,4,5,6,7,8,9,10])
passed = np.array([0,0,0,0,1,0,1,1,1,1])   # 1 = passed

from sklearn.linear_model import LogisticRegression
clf = LogisticRegression()
clf.fit(hours.reshape(-1,1), passed)

probs = clf.predict_proba(hours.reshape(-1,1))[:,1]   # probability of "passed"
preds = clf.predict(hours.reshape(-1,1))
print("hours -> P(pass) -> predicted class")
for h, p, c in zip(hours, probs, preds):
    print(f"  {h:2d}h  ->  {p:.3f}  ->  {c}")
print(f"\naccuracy: {accuracy_score(passed, preds):.2f}")

⚠️ **Common trap.** Logistic regression outputs a **probability**, not a class — you choose
the threshold (usually 0.5, but not always: e.g. for a rare-disease screen you might lower the
threshold to catch more true positives at the cost of more false alarms).

### ✏️ Your Turn 3.1
Predict `P(pass)` for someone who studied **4.5 hours** using the fitted `clf`.

In [ ]:
prob_4_5 = None
print(prob_4_5)

✅ **Solution**
```python
prob_4_5 = clf.predict_proba([[4.5]])[0,1]
```

---
## Chapter 4 — Decision Trees

📖 **Theory.** A decision tree asks a sequence of **yes/no questions** about features, splitting
data at each node to make the resulting groups as "pure" (single-class) as possible. Purity is
measured by **Gini impurity** or **entropy**:

```
Gini(node) = 1 - Σ(p_i)²        entropy(node) = -Σ p_i log2(p_i)
```
where `p_i` is the fraction of class `i` in that node. The tree picks, at each split, the
question that **most reduces** impurity.

🖼️ **Diagram — a simple tree**
```
              [hours < 5?]
             /            \\
          yes               no
          /                   \\
     [FAIL]              [attended class?]
                          /            \\
                       yes               no
                       /                   \\
                  [PASS]               [FAIL]
```

🧠 **Mental model.** A decision tree is a **flowchart** learned from data — at each node it finds
the single most informative question to ask next.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

# a slightly richer toy dataset: [hours_studied, attended_class] -> passed
X_tree = np.array([[1,0],[2,0],[3,1],[4,0],[5,1],[2,1],[8,0],[9,1],[6,1],[1,1]])
y_tree = np.array([0,0,0,0,1,1,0,1,1,0])

tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_tree, y_tree)
print(export_text(tree, feature_names=["hours","attended"]))
print("training accuracy:", accuracy_score(y_tree, tree.predict(X_tree)))

⚠️ **Common trap.** Unlimited-depth trees can grow one branch per training example —
**perfect** training accuracy, **terrible** generalization (severe overfitting). `max_depth`,
`min_samples_leaf`, and pruning are essential controls, not optional tuning.

### ✏️ Your Turn 4.1
Fit two trees on `X_tree, y_tree`: one with `max_depth=1` and one with `max_depth=None`
(unlimited). Compare their training accuracy — which one is more likely to overfit?

In [ ]:
shallow = None; deep = None
print(accuracy_score(y_tree, shallow.predict(X_tree)) if shallow else None)
print(accuracy_score(y_tree, deep.predict(X_tree)) if deep else None)

✅ **Solution**
```python
shallow = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_tree, y_tree)
deep = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X_tree, y_tree)
# deep likely reaches ~100% training accuracy -- a red flag for overfitting on such tiny data
```

---
## Chapter 5 — Ensembles: Random Forests & Boosting

📖 **Theory.** A single tree overfits easily. **Ensembles** combine many weak models into a
strong one:
- **Random Forest (bagging):** train many trees on **random subsets** of data and features,
  then **average/vote** their predictions. Reduces variance (overfitting) via diversity.
- **Gradient Boosting:** train trees **sequentially**, each one focused on correcting the
  previous trees' errors. Often more accurate, but more prone to overfitting if not tuned.

🖼️ **Diagram — bagging vs boosting**
```
 BAGGING (Random Forest):        BOOSTING (Gradient Boosting):
 tree1 ┐                         tree1 -> errors -> tree2 (fixes tree1's mistakes)
 tree2 ├─► vote/average               -> errors -> tree3 (fixes tree2's mistakes) -> ...
 tree3 ┘  (trained in parallel,      (trained sequentially, each depends on the last)
           on random subsets)
```


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

rf = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=0)
rf.fit(X_tree, y_tree)
print("Random Forest training accuracy:", accuracy_score(y_tree, rf.predict(X_tree)))

gb = GradientBoostingClassifier(n_estimators=50, max_depth=2, random_state=0)
gb.fit(X_tree, y_tree)
print("Gradient Boosting training accuracy:", accuracy_score(y_tree, gb.predict(X_tree)))

# feature importance -- which feature mattered most?
print("\nrandom forest feature importances (hours, attended):", rf.feature_importances_.round(3))

⚡ **Pro tip.** Random forests are a strong, low-maintenance **default** for tabular data —
robust to overfitting, needs little tuning. Reach for gradient boosting (XGBoost/LightGBM in
practice) when you need to squeeze out extra accuracy and can afford more careful tuning.

### ✏️ Your Turn 5.1
Train a `RandomForestClassifier` with `n_estimators=100` and print which feature (hours or
attended) has the higher importance.

In [ ]:
rf2 = None
more_important = None
print(more_important)

✅ **Solution**
```python
rf2 = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tree, y_tree)
more_important = "hours" if rf2.feature_importances_[0] > rf2.feature_importances_[1] else "attended"
```

---
## Chapter 6 — k-Nearest Neighbors (k-NN)

📖 **Theory.** k-NN has **no training phase** — it just memorizes the data. To classify a new
point, find its **k closest points** (by distance, usually Euclidean or cosine — from the NumPy
and Embeddings labs!) and take a **majority vote** of their labels.

🖼️ **Diagram — classifying by nearest neighbors**
```
        • class A
        •      ?  ← new point: 2 of its 3 nearest neighbors are A, 1 is B
      •    •  ○         -> vote: A wins (k=3)
   ○   ○
      ○  class B
```

🧠 **Mental model.** "You are the average of your 5 closest friends" — a point's predicted class
is decided by *voting among whoever is nearby*.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_tree, y_tree)
new_point = np.array([[4, 1]])   # 4 hours studied, attended class
pred = knn.predict(new_point)
neighbors_dist, neighbors_idx = knn.kneighbors(new_point)
print("predicted class:", pred[0])
print("3 nearest neighbors (by index):", neighbors_idx[0], "distances:", neighbors_dist[0].round(2))

⚠️ **Common trap.** k-NN is **sensitive to feature scale** — a feature ranging 0-1000 will
dominate distance calculations over one ranging 0-1, even if the small-range feature is more
predictive. Always **normalize/standardize features** before using k-NN (same lesson as
normalizing inputs in the PyTorch/TF labs).

### ✏️ Your Turn 6.1
Predict the class of a new point `[7, 0]` (7 hours, didn't attend) using `k=1` vs `k=5`. Do they
agree?

In [ ]:
pred_k1 = None
pred_k5 = None
print(pred_k1, pred_k5)

✅ **Solution**
```python
pred_k1 = KNeighborsClassifier(n_neighbors=1).fit(X_tree, y_tree).predict([[7,0]])
pred_k5 = KNeighborsClassifier(n_neighbors=5).fit(X_tree, y_tree).predict([[7,0]])
```

---
## Chapter 7 — k-Means Clustering (Unsupervised)

📖 **Theory.** k-Means is **unsupervised** — no labels at all. Given `k` (number of clusters),
it iterates:
1. Assign each point to its **nearest centroid**.
2. Move each centroid to the **mean** of its assigned points.
3. Repeat until centroids stop moving.

🖼️ **Diagram — the k-Means loop**
```
 init random centroids ──► assign points to nearest centroid ──► recompute centroid = mean
        ▲                                                                    │
        └────────────────────── repeat until stable ◄─────────────────────────┘
```


In [ ]:
from sklearn.cluster import KMeans

np.random.seed(1)
cluster_a = np.random.randn(20, 2) + [0, 0]
cluster_b = np.random.randn(20, 2) + [8, 8]
cluster_c = np.random.randn(20, 2) + [0, 8]
X_cluster = np.vstack([cluster_a, cluster_b, cluster_c])

km = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = km.fit_predict(X_cluster)
print("cluster assignment counts:", np.bincount(labels))
print("cluster centers:\n", km.cluster_centers_.round(2))

⚠️ **Common trap.** You must **choose k in advance** — k-Means won't discover the "right"
number of clusters for you. The **elbow method** (plotting inertia vs. k and looking for a bend)
is a common heuristic for picking k.

### ✏️ Your Turn 7.1
Run k-Means with `n_clusters=2` on `X_cluster` (which really has 3 natural clusters). What
happens — does it merge two of the true clusters together?

In [ ]:
km2 = None
print(np.bincount(km2.labels_) if km2 else None)

✅ **Solution**
```python
km2 = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X_cluster)
# With only 2 clusters allowed, two of the three true groups get merged into one.
```

---
## Chapter 8 — Support Vector Machines (concept)

📖 **Theory.** An SVM finds the **decision boundary that maximizes the margin** — the distance
to the nearest points of each class ("support vectors"). A wider margin tends to generalize
better. For data that isn't linearly separable, the **kernel trick** implicitly projects data
into a higher dimension where it *is* separable — without ever computing that projection
explicitly (a clever piece of math beyond this lab's scope, but the intuition matters).

🖼️ **Diagram — maximum margin**
```
   class A    │  margin  │    class B
      •   •   │◄────────►│   ○   ○
        •     │  boundary │      ○
      •       │    ↑      │        ○
              the widest possible gap between classes
```


In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel="linear")
svm.fit(X_tree, y_tree)
print("linear SVM training accuracy:", accuracy_score(y_tree, svm.predict(X_tree)))
print("number of support vectors:", svm.n_support_)

# kernel trick: for non-linearly-separable data, an RBF kernel can still separate it
from sklearn.datasets import make_circles
Xc, yc = make_circles(n_samples=100, noise=0.05, factor=0.3, random_state=0)
svm_linear = SVC(kernel="linear").fit(Xc, yc)
svm_rbf = SVC(kernel="rbf").fit(Xc, yc)
print(f"\ncircular data -- linear kernel acc: {accuracy_score(yc, svm_linear.predict(Xc)):.2f}")
print(f"circular data -- RBF kernel acc:    {accuracy_score(yc, svm_rbf.predict(Xc)):.2f}")

⚡ **Pro tip.** The RBF (radial basis function) kernel is a strong default when you don't know
whether your data is linearly separable — it can flexibly capture non-linear boundaries.

### ✏️ Your Turn 8.1
On the `make_circles` data, try `kernel="poly"` and compare its accuracy to `"rbf"`.

In [ ]:
svm_poly = None
print(accuracy_score(yc, svm_poly.predict(Xc)) if svm_poly else None)

✅ **Solution**
```python
svm_poly = SVC(kernel="poly", degree=3).fit(Xc, yc)
print(accuracy_score(yc, svm_poly.predict(Xc)))
```

---
## Chapter 9 — Dimensionality Reduction: PCA

📖 **Theory.** Real datasets often have **many correlated features**. **Principal Component
Analysis (PCA)** finds new axes (**principal components**) that capture the most **variance** in
the data, letting you compress many features into a few — for visualization, noise reduction, or
speeding up downstream models. It's built directly on **eigenvectors/eigenvalues** from the
linear algebra you learned in the NumPy lab.

🖼️ **Diagram — PCA finds the "spread-out" direction**
```
 original 2D data (correlated):        after PCA (rotated to new axes):
   y                                      PC2
   │  •  •                                │
   │    •  •  •                           │  •  •  •  •  •  •     ← PC1 captures
   │  •    •  •         ──rotate──►       │                          most of the
   │•  •  •                               │                          spread
   └──────────► x                         └──────────────► PC1
```


In [ ]:
from sklearn.decomposition import PCA

np.random.seed(2)
# correlated 2D data: y roughly = 2x + noise
x1 = np.random.randn(100)
x2 = 2*x1 + np.random.randn(100)*0.3
X_corr = np.column_stack([x1, x2])

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_corr)
print("explained variance ratio per component:", pca.explained_variance_ratio_.round(3))
print("-> PC1 alone captures", f"{pca.explained_variance_ratio_[0]:.1%}", "of the variance")

# reduce to just 1 dimension, keeping most of the information
pca_1d = PCA(n_components=1)
X_reduced = pca_1d.fit_transform(X_corr)
print("\noriginal shape:", X_corr.shape, "-> reduced shape:", X_reduced.shape)

⚠️ **Common trap.** PCA is sensitive to feature **scale**, just like k-NN — always
standardize features (zero mean, unit variance) before PCA, or a large-scale feature will
dominate the "variance" the components chase, even if it's not actually more informative.

### ✏️ Your Turn 9.1
Apply PCA to reduce the Iris dataset (4 features) to 2 components, and report the total
variance explained by those 2 components.

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
pca_iris = None
total_variance_explained = None
print(total_variance_explained)

✅ **Solution**
```python
pca_iris = PCA(n_components=2).fit(iris.data)
total_variance_explained = pca_iris.explained_variance_ratio_.sum()
# ~0.977 -- 2 components capture ~98% of the original 4 features' variance
```

---
## 🏆 Chapter 10 — Capstone: A Full Classical-ML Pipeline

Build a complete pipeline on the **breast cancer** dataset: split data, standardize features,
train **three** models (Logistic Regression, Random Forest, k-NN), evaluate each on the test
set, and report which performed best. This mirrors a real model-selection workflow.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, data.target
print("dataset shape:", X.shape, "classes:", np.unique(y))

### ✏️ Capstone Tasks
1. Split into train/test (80/20, `random_state=0`).
2. **Standardize** features using training-set statistics (`StandardScaler`).
3. Train `LogisticRegression`, `RandomForestClassifier`, and `KNeighborsClassifier(n_neighbors=5)`.
4. Evaluate each on the **test** set with `accuracy_score`.
5. Report which model performed best.

In [ ]:
# Your pipeline here


✅ **Capstone Solution**
```python
# 1. split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# 2. standardize (fit on TRAIN only, apply to both)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 3. train three models
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=0),
    "kNN": KNeighborsClassifier(n_neighbors=5),
}
for name, model in models.items():
    model.fit(X_train_s, y_train)

# 4. evaluate
results = {name: accuracy_score(y_test, model.predict(X_test_s)) for name, model in models.items()}
for name, acc in results.items():
    print(f"{name:20s} test accuracy: {acc:.3f}")

# 5. best model
best = max(results, key=results.get)
print(f"\\nBest model: {best} ({results[best]:.3f})")
```

🎉 **You've mastered classical ML!** Linear/logistic regression, decision trees, ensembles
(bagging & boosting), k-NN, k-Means, SVMs, and PCA — with the math and mental models behind
each, not just API calls. These algorithms remain the right choice for most tabular-data
problems, and this same train/standardize/evaluate/compare workflow is exactly how real model
selection is done in production.

---
### 📌 Concept Quick-Reference
**Regression:** closed-form Normal Equation `(XᵀX)⁻¹Xᵀy`, or gradient descent; minimize MSE
**Logistic regression:** sigmoid squashes to (0,1); minimize log loss (cross-entropy)
**Decision trees:** greedy splits minimizing Gini/entropy impurity; control depth to avoid overfit
**Ensembles:** bagging (Random Forest, parallel + average) vs boosting (sequential, fix errors)
**k-NN:** no training; classify by majority vote of k nearest points; scale features first
**k-Means:** unsupervised; alternate assign-to-nearest-centroid and recompute-centroid-as-mean
**SVM:** maximize margin between classes; kernel trick for non-linear separability
**PCA:** eigenvectors of the covariance matrix; new axes capturing max variance; scale first
**Workflow:** split -> standardize (fit on train only) -> train -> evaluate -> compare
